# LeetCode 2855 — Minimum Right Shifts to Sort the Array bằng Python

Link đề: https://leetcode.com/problems/minimum-right-shifts-to-sort-the-array/ · Độ khó: **Easy** · Chủ đề: Array, Rotation (xoay mảng)

Skeleton LeetCode:

```python
from typing import List

class Solution:
    def minimumRightShifts(self, nums: List[int]) -> int:
        ...
```

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: **dịch phải (right shift)** một mảng nghĩa là phần tử ở chỉ số `i` chuyển sang chỉ số `(i + 1) % n` (phần tử cuối chuyển về đầu). Tìm **số lần dịch phải tối thiểu** để mảng được sắp xếp **tăng dần**, hoặc `-1` nếu không thể.
2. Biết **tự tính tay**: thực hiện từng lần dịch, nhìn mảng "xoay vòng" và nhận ra khi nào nó đã sắp xếp.
3. Học các công cụ Python: cắt lát và nối list `a[n - k:] + a[:n - k]`, `list.index`, `min`, toán tử `%` cho chỉ số vòng tròn, `zip` / `all` để kiểm "đã sắp xếp", `sorted`.
4. Đi từ **Brute Force: thử dịch `0, 1, ..., n − 1` lần** → **dùng vị trí phần tử nhỏ nhất** → **đếm số chỗ "tụt xuống" trên vòng tròn** → **tìm điểm gãy duy nhất rồi kiểm tra**.
5. Hiểu vì sao mảng **chỉ sắp xếp được bằng dịch vòng** khi nó là **một mảng tăng dần bị xoay**, nên có **tối đa một** chỗ giảm khi đi vòng tròn; và tránh các lỗi kinh điển: nhầm dịch phải với dịch trái, quên trường hợp đã sắp xếp (trả `0`), quên so cặp `nums[n − 1]` với `nums[0]`.

## 2. Đề bài & ví dụ

Bạn được cho một mảng `nums` có độ dài `n` gồm các số nguyên **dương, đôi một khác nhau**.

Một **lần dịch phải** (right shift) được định nghĩa là: dịch phần tử ở chỉ số `i` sang chỉ số `(i + 1) % n`, với mọi `i`. Nói cách khác, mọi phần tử nhích sang phải một ô và phần tử cuối cùng chuyển về đầu mảng.

Trả về **số lần dịch phải tối thiểu** cần thiết để **sắp xếp** `nums` (tăng dần), hoặc `-1` nếu **không thể**.

| Ví dụ | `nums` | Output | Giải thích |
|---|---|---|---|
| 1 | `[3,4,5,1,2]` | `2` | Dịch lần 1: `[2,3,4,5,1]`. Dịch lần 2: `[1,2,3,4,5]` đã sắp xếp. Cần 2 lần. |
| 2 | `[1,3,5]` | `0` | Mảng đã được sắp xếp, cần 0 lần dịch. |
| 3 | `[2,1,4]` | `-1` | Dù dịch bao nhiêu lần cũng không sắp xếp được. |

Constraints:

- `1 <= nums.length <= 100`
- `1 <= nums[i] <= 100`
- `nums` gồm các số **đôi một khác nhau**.

In [1]:
# Chạy thử "bằng tay" ví dụ 1: dịch phải từng lần
nums = [3, 4, 5, 1, 2]
cur = nums[:]
for k in range(1, 4):
    cur = [cur[-1]] + cur[:-1]        # phần tử cuối chuyển về đầu
    print(f"sau {k} lần dịch:", cur)

sau 1 lần dịch: [2, 3, 4, 5, 1]
sau 2 lần dịch: [1, 2, 3, 4, 5]
sau 3 lần dịch: [5, 1, 2, 3, 4]


## 3. Kiến thức nền

### 3.1. Dịch phải một lần bằng cắt lát

Với list `a`, `a[-1]` là phần tử cuối, `a[:-1]` là mọi phần tử trừ phần tử cuối. Dịch phải một lần:

`a = [a[-1]] + a[:-1]`

Ví dụ `[1, 2, 3, 4]` → `[4, 1, 2, 3]`: `4` (phần tử cuối) lên đầu, các phần tử còn lại nhích sang phải.

In [2]:
a = [1, 2, 3, 4]
print([a[-1]] + a[:-1])      # [4, 1, 2, 3]
print(a[-1], a[:-1])         # 4 [1, 2, 3]

[4, 1, 2, 3]
4 [1, 2, 3]


### 3.2. Dịch phải `k` lần bằng một phép cắt lát

Dịch phải `k` lần (với `0 <= k <= n`) tương đương đem **`k` phần tử cuối** lên đầu: `a[n − k:] + a[:n − k]`. Ví dụ `[3,4,5,1,2]` dịch 2 lần: `k = 2`, `a[3:] = [1, 2]`, `a[:3] = [3, 4, 5]` ⇒ `[1, 2, 3, 4, 5]`.

In [3]:
def rotate_right(a, k):
    n = len(a)
    k %= n                    # k lần dịch tương đương k mod n lần (xoay vòng)
    return a[n - k:] + a[:n - k]

a = [3, 4, 5, 1, 2]
print(rotate_right(a, 2))     # [1, 2, 3, 4, 5]
print(rotate_right(a, 0))     # [3, 4, 5, 1, 2]
print(rotate_right(a, 5))     # [3, 4, 5, 1, 2]  (dịch n lần = quay về)

[1, 2, 3, 4, 5]
[3, 4, 5, 1, 2]
[3, 4, 5, 1, 2]


### 3.3. Chỉ số vòng tròn bằng `%`

`(i + 1) % n` cho chỉ số của phần tử **kế tiếp theo vòng tròn**: với `i = n − 1` ta được `0`. Dùng để so sánh `nums[n − 1]` với `nums[0]` mà không cần viết riêng trường hợp.

In [4]:
n = 5
print([(i + 1) % n for i in range(n)])     # [1, 2, 3, 4, 0]

[1, 2, 3, 4, 0]


### 3.4. `min`, `list.index`

`min(a)` là giá trị nhỏ nhất; `a.index(x)` là chỉ số của **lần xuất hiện đầu tiên** của `x` (báo `ValueError` nếu không có). Vì các số **đôi một khác nhau**, `a.index(min(a))` là vị trí duy nhất của phần tử nhỏ nhất.

In [5]:
a = [3, 4, 5, 1, 2]
print(min(a), a.index(min(a)))      # 1 3

1 3


### 3.5. `zip` và `all` để kiểm tra "đã sắp xếp"

`zip(a, a[1:])` ghép các cặp kề nhau `(a[0], a[1]), (a[1], a[2]), ...`. `all(x < y for x, y in zip(a, a[1:]))` kiểm mọi cặp kề đều tăng ngặt. Vì các số khác nhau nên "không giảm" cũng là "tăng ngặt".

In [6]:
a = [1, 2, 3]
print(list(zip(a, a[1:])))                       # [(1, 2), (2, 3)]
print(all(x < y for x, y in zip(a, a[1:])))      # True
b = [1, 3, 2]
print(all(x < y for x, y in zip(b, b[1:])))      # False

[(1, 2), (2, 3)]
True
False


### 3.6. `sorted` và so sánh list

`sorted(a)` là bản sắp xếp tăng dần. Hai list bằng nhau (`==`) khi cùng độ dài và từng phần tử giống nhau.

In [7]:
print(sorted([3, 4, 5, 1, 2]))                        # [1, 2, 3, 4, 5]
print(rotate_right([3, 4, 5, 1, 2], 2) == sorted([3, 4, 5, 1, 2]))    # True

[1, 2, 3, 4, 5]
True


## 4. Ý tưởng nền tảng / cách nghĩ trực giác

Hình dung mảng như một **vòng tròn** các số. Dịch phải chỉ **xoay** vòng tròn, không đổi thứ tự quanh vòng. Mảng sắp xếp được ⇔ có một điểm cắt trên vòng tròn sao cho đọc từ đó theo chiều kim đồng hồ ta được dãy **tăng dần**.

Ví dụ `[3,4,5,1,2]` trên vòng tròn: `3 → 4 → 5 → 1 → 2 → (về 3)`. Đi một vòng, ta thấy các bước **tăng** (`3→4`, `4→5`, `1→2`, `2→3`) và đúng **một bước giảm** (`5→1`). Chỗ giảm là nơi "cắt" vòng tròn: bắt đầu đọc từ `1` thì được `1, 2, 3, 4, 5`.

Ví dụ `[2,1,4]`: `2 → 1` giảm, `1 → 4` tăng, `4 → 2` giảm: **hai** chỗ giảm ⇒ không có điểm cắt nào cho dãy tăng ⇒ `-1`.

**Quy tắc:** trên vòng tròn, số chỗ giảm phải bằng **đúng 1** (với `n >= 2`; vì các số khác nhau, mỗi cặp kề hoặc tăng hoặc giảm, và vòng tròn không thể toàn tăng). Khi đó, nếu chỗ giảm nằm giữa `nums[i]` và `nums[(i + 1) % n]`, phần tử nhỏ nhất ở chỉ số `(i + 1) % n`, và cần dịch phải `n − (i + 1)` lần (đưa nó về chỉ số `0`), tính theo mod `n`.

Với `[3,4,5,1,2]`: chỗ giảm giữa `i = 2` và `3`, `n − (2 + 1) = 5 − 3 = 2` ✓.

In [8]:
nums = [3, 4, 5, 1, 2]
n = len(nums)
for i in range(n):
    j = (i + 1) % n
    print(f"{nums[i]} -> {nums[j]}:", "GIẢM" if nums[i] > nums[j] else "tăng")

3 -> 4: tăng
4 -> 5: tăng
5 -> 1: GIẢM
1 -> 2: tăng
2 -> 3: tăng


## 5. Cách 1 — Brute Force / baseline: thử dịch `0, 1, ..., n − 1` lần

**Ý tưởng:** với `k` từ `0` đến `n − 1`: tạo mảng sau `k` lần dịch phải (`rotate_right`), kiểm tra đã sắp xếp chưa. `k` đầu tiên thoả thì trả về `k` (vì ta thử tăng dần nên là nhỏ nhất). Không có `k` nào ⇒ `-1`.

Chỉ cần thử `k < n`: dịch `n` lần là quay về mảng ban đầu.

In [9]:
from typing import List

def is_sorted(a: List[int]) -> bool:
    return all(a[i] < a[i + 1] for i in range(len(a) - 1))

def min_shifts_bruteforce(nums: List[int]) -> int:
    n = len(nums)
    for k in range(n):
        if is_sorted(rotate_right(nums, k)):
            return k
    return -1

In [10]:
print(min_shifts_bruteforce([3, 4, 5, 1, 2]))     # 2
print(min_shifts_bruteforce([1, 3, 5]))           # 0
print(min_shifts_bruteforce([2, 1, 4]))           # -1
print(min_shifts_bruteforce([7]))                 # 0 (một phần tử luôn đã sắp xếp)

2
0
-1
0


**Độ phức tạp:** gọi `n = len(nums)`. Có `n` giá trị `k`, mỗi giá trị tạo mảng mới `O(n)` và kiểm tra `O(n)` ⇒ Time `O(n²)`. Space `O(n)`.

**Nút thắt:** ta tạo và kiểm tra `n` mảng gần như giống nhau. Nhưng chỉ có **một** `k` có thể đúng (đưa phần tử nhỏ nhất về đầu). Ta chỉ cần thử **một** ứng viên.

## 6. Ý tưởng thuật toán / bước đột phá

**Đột phá 1 — chỉ có một ứng viên `k`.** Để sắp xếp tăng dần, phần tử nhỏ nhất phải về chỉ số `0`. Nếu nó đang ở chỉ số `p`, mỗi lần dịch phải đẩy nó thêm một ô; để về chỉ số `0` (qua vòng) cần `k = (n − p) % n` lần. Đó là **ứng viên duy nhất**. Chỉ cần dịch `k` lần rồi kiểm tra đã sắp xếp chưa.

**Đột phá 2 — không cần dịch thật, đếm chỗ giảm trên vòng tròn.** Mảng sắp xếp được bằng dịch vòng ⇔ trên vòng tròn có **đúng một** cặp `(nums[i], nums[(i + 1) % n])` với `nums[i] > nums[(i + 1) % n]`. Khi đó phần tử nhỏ nhất ở chỉ số `(i + 1) % n` và `k = n − (i + 1)` (bằng `0` nếu `i = n − 1`, tức đã sắp xếp).

**Đột phá 3 — hai "đoạn tăng" ghép lại.** Khi có đúng một chỗ gãy ở `i`, mảng gồm hai đoạn tăng `nums[0..i]` và `nums[i + 1..n − 1]`; để dịch vòng thành mảng tăng, cần `nums[n − 1] < nums[0]` (đuôi của đoạn sau nhỏ hơn đầu đoạn trước). Điều kiện này chính là "cặp vòng `(n − 1, 0)` **không** giảm khi chỗ gãy đã nằm ở giữa".

In [11]:
nums = [3, 4, 5, 1, 2]
n = len(nums)
p = nums.index(min(nums))
k = (n - p) % n
print("phần tử nhỏ nhất ở chỉ số", p, "-> ứng viên k =", k)
print("sau k lần dịch:", rotate_right(nums, k), "| đã sắp xếp?", is_sorted(rotate_right(nums, k)))

phần tử nhỏ nhất ở chỉ số 3 -> ứng viên k = 2
sau k lần dịch: [1, 2, 3, 4, 5] | đã sắp xếp? True


## 7. Các cách tiếp theo

Ta giảng thêm 3 cách nữa.

### Cách 2 — Vị trí phần tử nhỏ nhất, dịch đúng `k` lần rồi kiểm tra

**Từng đoạn:**

- `p = nums.index(min(nums))`.
- `k = (n − p) % n`: số lần dịch đưa phần tử nhỏ nhất về đầu (nếu `p = 0` thì `k = 0`).
- Dịch `k` lần (bằng cắt lát) rồi kiểm đã sắp xếp: nếu có ⇒ `k`, nếu không ⇒ `-1`.

In [12]:
def min_shifts_min_index(nums: List[int]) -> int:
    n = len(nums)
    p = nums.index(min(nums))
    k = (n - p) % n
    rotated = nums[n - k:] + nums[:n - k]
    return k if is_sorted(rotated) else -1

In [13]:
print(min_shifts_min_index([3, 4, 5, 1, 2]))     # 2
print(min_shifts_min_index([1, 3, 5]))           # 0
print(min_shifts_min_index([2, 1, 4]))           # -1

2
0
-1


**Code nộp LeetCode (Cách 2):**

```python
from typing import List

class Solution:
    def minimumRightShifts(self, nums: List[int]) -> int:
        n = len(nums)
        p = nums.index(min(nums))
        k = (n - p) % n
        rotated = nums[n - k:] + nums[:n - k]
        if all(rotated[j] < rotated[j + 1] for j in range(n - 1)):
            return k
        return -1
```

**Độ phức tạp:** `min`, `index`, cắt lát và kiểm tra từng cặp kề đều `O(n)` ⇒ Time `O(n)`, Space `O(n)` cho mảng xoay.

### Cách 3 — Đếm số chỗ giảm trên vòng tròn (không dịch thật)

**Từng đoạn:**

- Duyệt `i` từ `0` đến `n − 1`; so `nums[i]` với `nums[(i + 1) % n]`.
- Mỗi lần `nums[i] > nums[(i + 1) % n]` ⇒ ghi nhận `descents += 1` và nhớ `pos = i`.
- `descents == 0` chỉ xảy ra khi `n = 1` ⇒ trả `0`. `descents >= 2` ⇒ `-1`. `descents == 1` ⇒ `n − (pos + 1)`.

In [14]:
def min_shifts_circular(nums: List[int]) -> int:
    n = len(nums)
    descents = 0
    pos = -1
    for i in range(n):
        if nums[i] > nums[(i + 1) % n]:
            descents += 1
            pos = i
    if descents == 0:          # chỉ xảy ra khi n == 1
        return 0
    if descents > 1:
        return -1
    return n - (pos + 1)

In [15]:
print(min_shifts_circular([3, 4, 5, 1, 2]))     # 2
print(min_shifts_circular([1, 3, 5]))           # 0  (chỗ giảm duy nhất là cặp vòng 5 -> 1, pos = 2 = n - 1)
print(min_shifts_circular([2, 1, 4]))           # -1
print(min_shifts_circular([5]))                 # 0

2
0
-1
0


**Code nộp LeetCode (Cách 3):**

```python
from typing import List

class Solution:
    def minimumRightShifts(self, nums: List[int]) -> int:
        n = len(nums)
        descents = 0
        pos = -1
        for i in range(n):
            if nums[i] > nums[(i + 1) % n]:
                descents += 1
                pos = i
        if descents == 0:
            return 0
        if descents > 1:
            return -1
        return n - (pos + 1)
```

**Độ phức tạp:** một lượt duyệt ⇒ Time `O(n)`, Space `O(1)`.

### Cách 4 — Tìm điểm gãy đầu tiên, kiểm tra hai đoạn

**Từng đoạn:**

- Tìm `i` đầu tiên với `nums[i] > nums[i + 1]` (trong khoảng `0..n − 2`). Không có ⇒ mảng đã tăng ⇒ `0`.
- Nếu có: kiểm `nums[i + 1:] + nums[:i + 1]` đã sắp xếp. Có ⇒ `n − (i + 1)`; không ⇒ `-1`.
- Đây là bản "tìm điểm cắt rồi xác minh": dễ nghĩ cho người chưa quen đếm chỗ giảm vòng tròn.

In [16]:
def min_shifts_break(nums: List[int]) -> int:
    n = len(nums)
    for i in range(n - 1):
        if nums[i] > nums[i + 1]:               # điểm gãy đầu tiên
            rotated = nums[i + 1:] + nums[:i + 1]   # đọc từ phần tử nhỏ nhất
            return n - (i + 1) if is_sorted(rotated) else -1
    return 0                                     # không có điểm gãy: đã tăng dần

In [17]:
print(min_shifts_break([3, 4, 5, 1, 2]))     # 2
print(min_shifts_break([1, 3, 5]))           # 0
print(min_shifts_break([2, 1, 4]))           # -1
print(min_shifts_break([4, 5, 1, 2, 3]))     # 3

2
0
-1
3


**Code nộp LeetCode (Cách 4):**

```python
from typing import List

class Solution:
    def minimumRightShifts(self, nums: List[int]) -> int:
        n = len(nums)
        for i in range(n - 1):
            if nums[i] > nums[i + 1]:
                rotated = nums[i + 1:] + nums[:i + 1]
                if all(rotated[j] < rotated[j + 1] for j in range(n - 1)):
                    return n - (i + 1)
                return -1
        return 0
```

**Độ phức tạp:** Time `O(n)`, Space `O(n)` cho mảng xoay.

## 8. TRACE từng bước

Trace Cách 3 (đếm chỗ giảm vòng tròn) trên `nums = [3, 4, 5, 1, 2]` và `[2, 1, 4]`.

Bảng cho `[3,4,5,1,2]` (`n = 5`):

| `i` | `nums[i]` | `(i+1) % n` | `nums[(i+1) % n]` | giảm? | `descents` | `pos` |
|---|---|---|---|---|---|---|
| 0 | 3 | 1 | 4 | không | 0 | −1 |
| 1 | 4 | 2 | 5 | không | 0 | −1 |
| 2 | 5 | 3 | 1 | **có** | 1 | 2 |
| 3 | 1 | 4 | 2 | không | 1 | 2 |
| 4 | 2 | 0 | 3 | không | 1 | 2 |

`descents == 1`, `pos = 2` ⇒ đáp án `n − (pos + 1) = 5 − 3 = 2` ✓.

In [18]:
def trace_circular(nums):
    n = len(nums)
    descents, pos = 0, -1
    print(f"nums={nums}, n={n}")
    for i in range(n):
        j = (i + 1) % n
        giam = nums[i] > nums[j]
        if giam:
            descents += 1
            pos = i
        print(f"i={i}: nums[{i}]={nums[i]} vs nums[{j}]={nums[j]} -> {'GIẢM' if giam else 'tăng'} | descents={descents}, pos={pos}")
    if descents == 0:
        kq = 0
    elif descents > 1:
        kq = -1
    else:
        kq = n - (pos + 1)
    print("kết quả:", kq)
    return kq

trace_circular([3, 4, 5, 1, 2])
print("---")
trace_circular([2, 1, 4])
print("---")
trace_circular([1, 3, 5])

nums=[3, 4, 5, 1, 2], n=5
i=0: nums[0]=3 vs nums[1]=4 -> tăng | descents=0, pos=-1
i=1: nums[1]=4 vs nums[2]=5 -> tăng | descents=0, pos=-1
i=2: nums[2]=5 vs nums[3]=1 -> GIẢM | descents=1, pos=2
i=3: nums[3]=1 vs nums[4]=2 -> tăng | descents=1, pos=2
i=4: nums[4]=2 vs nums[0]=3 -> tăng | descents=1, pos=2
kết quả: 2
---
nums=[2, 1, 4], n=3
i=0: nums[0]=2 vs nums[1]=1 -> GIẢM | descents=1, pos=0
i=1: nums[1]=1 vs nums[2]=4 -> tăng | descents=1, pos=0
i=2: nums[2]=4 vs nums[0]=2 -> GIẢM | descents=2, pos=2
kết quả: -1
---
nums=[1, 3, 5], n=3
i=0: nums[0]=1 vs nums[1]=3 -> tăng | descents=0, pos=-1
i=1: nums[1]=3 vs nums[2]=5 -> tăng | descents=0, pos=-1
i=2: nums[2]=5 vs nums[0]=1 -> GIẢM | descents=1, pos=2
kết quả: 0


0

In [19]:
# Trace brute force: xem từng mảng sau k lần dịch
def trace_bruteforce(nums):
    n = len(nums)
    for k in range(n):
        r = rotate_right(nums, k)
        print(f"k={k}: {r} -> {'ĐÃ SẮP XẾP' if is_sorted(r) else 'chưa'}")
        if is_sorted(r):
            return k
    return -1

print(trace_bruteforce([3, 4, 5, 1, 2]))
print(trace_bruteforce([2, 1, 4]))

k=0: [3, 4, 5, 1, 2] -> chưa
k=1: [2, 3, 4, 5, 1] -> chưa
k=2: [1, 2, 3, 4, 5] -> ĐÃ SẮP XẾP
2
k=0: [2, 1, 4] -> chưa
k=1: [4, 2, 1] -> chưa
k=2: [1, 4, 2] -> chưa
-1


## 9. Code nộp LeetCode

Dùng Cách 3 (đếm chỗ giảm trên vòng tròn): `O(n)` thời gian, `O(1)` bộ nhớ.

In [20]:
from typing import List

class Solution:
    def minimumRightShifts(self, nums: List[int]) -> int:
        n = len(nums)
        descents = 0                      # số chỗ giảm trên vòng tròn
        pos = -1                          # chỉ số i của chỗ giảm (nums[i] > nums[(i + 1) % n])
        for i in range(n):
            if nums[i] > nums[(i + 1) % n]:
                descents += 1
                pos = i
        if descents == 0:                 # n == 1
            return 0
        if descents > 1:                  # không xoay thành mảng tăng được
            return -1
        return n - (pos + 1)              # đưa phần tử nhỏ nhất (chỉ số pos + 1) về đầu

In [21]:
print(Solution().minimumRightShifts([3, 4, 5, 1, 2]))     # 2
print(Solution().minimumRightShifts([1, 3, 5]))           # 0
print(Solution().minimumRightShifts([2, 1, 4]))           # -1

2
0
-1


## 10. Vì sao thuật toán đúng

**Bước 1 — dịch phải chỉ xoay vòng.** Dịch phải một lần chuyển `nums[i]` sang chỉ số `(i + 1) % n`. Thứ tự các phần tử **quanh vòng tròn** không đổi; chỉ điểm bắt đầu đọc thay đổi. Sau `k` lần dịch, mảng là `nums[n − k:] + nums[:n − k]`.

**Bước 2 — điều kiện sắp xếp được.** Mảng sau xoay là mảng tăng ⇔ đọc quanh vòng tròn từ phần tử nhỏ nhất theo chiều tăng chỉ số ta thấy dãy tăng dần. Nghĩa là trong `n` cặp vòng `(nums[i], nums[(i + 1) % n])`, **chỉ cặp đứng ngay trước phần tử nhỏ nhất** là giảm (từ phần tử lớn nhất sang phần tử nhỏ nhất), còn `n − 1` cặp còn lại tăng.

**Bước 3 — đếm chỗ giảm.**

- Với `n >= 2` và các số khác nhau, vòng tròn không thể có `0` chỗ giảm (nếu mọi cặp vòng đều tăng thì `a₀ < a₁ < ... < a_{n−1} < a₀` mâu thuẫn). Vậy `descents >= 1`.
- Nếu `descents = 1`: tại chỗ giảm `(i, (i + 1) % n)`, phần tử `nums[(i + 1) % n]` là nhỏ nhất (nó nhỏ hơn phần tử đứng trước, và mọi cặp còn lại đều tăng nên nó là điểm đáy của vòng). Đọc từ đó cho dãy tăng ⇒ sắp xếp được.
- Nếu `descents >= 2`: một dãy tăng khi xoay có đúng một chỗ giảm vòng, mà số chỗ giảm vòng **không đổi** khi xoay (xoay chỉ đổi điểm bắt đầu đọc), nên mảng không thể xoay thành mảng tăng (mảng tăng có `descents = 1` ở cặp `(n − 1, 0)` — với `n >= 2`) ⇒ `-1`.
- Với `n = 1`: `nums[0] > nums[0]` sai ⇒ `descents = 0`, mảng một phần tử luôn đã sắp xếp ⇒ `0`.

**Bước 4 — số lần dịch.** Giả sử `descents = 1` tại `i`, phần tử nhỏ nhất ở chỉ số `m = (i + 1) % n`. Muốn đưa nó về chỉ số `0`, mỗi lần dịch phải tăng chỉ số một đơn vị theo mod `n`: cần `k ≡ −m (mod n)`, nhỏ nhất `k = (n − m) % n`.

- Nếu `i < n − 1`: `m = i + 1`, `k = (n − i − 1) % n = n − (i + 1)` (vì `1 <= i + 1 <= n − 1`).
- Nếu `i = n − 1`: `m = 0`, `k = 0 = n − (i + 1)`. Công thức `n − (pos + 1)` đúng cả hai trường hợp.

**Tính nhỏ nhất:** vì `0 <= k < n` và phần tử nhỏ nhất phải về chỉ số `0`, `k` này là duy nhất trong `[0, n)`; các `k + n, k + 2n, ...` lớn hơn.

## 11. Phân tích Time / Space Complexity

`n = len(nums)`.

| Cách | Time | Space | Ghi chú |
|---|---|---|---|
| 1. Thử mọi `k` | `O(n²)` | `O(n)` | oracle |
| 2. Vị trí min + xoay + kiểm tra | `O(n)` | `O(n)` | một ứng viên duy nhất |
| 3. Đếm chỗ giảm vòng tròn | `O(n)` | `O(1)` | **code nộp** |
| 4. Điểm gãy đầu tiên + kiểm tra | `O(n)` | `O(n)` | dễ nghĩ |

Với `n <= 100` mọi cách tức thì; cách 3 là tối ưu về bộ nhớ.

## 12. Bộ test

### 12.1. Ví dụ của đề và edge case

Các ca quan trọng: đã sắp xếp (`0`); đảo hoàn toàn (`[3,2,1]` không xoay được); một phần tử; hai phần tử (đã sắp xếp / đảo); phần tử nhỏ nhất ở cuối (cần `1` lần); phần tử nhỏ nhất ở vị trí 1 (cần `n − 1` lần); có hai chỗ giảm.

In [22]:
impls = {
    "min_index": min_shifts_min_index,
    "circular": min_shifts_circular,
    "break": min_shifts_break,
    "Solution": Solution().minimumRightShifts,
}
cases = [
    ("ví dụ 1", [3, 4, 5, 1, 2], 2),
    ("ví dụ 2", [1, 3, 5], 0),
    ("ví dụ 3", [2, 1, 4], -1),
    ("một phần tử", [7], 0),
    ("hai phần tử đã sắp xếp", [1, 2], 0),
    ("hai phần tử đảo", [2, 1], 1),
    ("nhỏ nhất ở cuối", [2, 3, 4, 1], 1),
    ("nhỏ nhất ở vị trí 1", [5, 1, 2, 3, 4], 4),
    ("đảo hoàn toàn", [3, 2, 1], -1),
    ("hai chỗ giảm", [3, 1, 4, 2], -1),
    ("xoay giữa", [4, 5, 1, 2, 3], 3),
]
for name, nums, exp in cases:
    assert min_shifts_bruteforce(list(nums)) == exp, ("bruteforce", name)
    for k, f in impls.items():
        got = f(list(nums))
        assert got == exp, (name, k, got, exp)
    print("PASS", name, "->", exp)

PASS ví dụ 1 -> 2
PASS ví dụ 2 -> 0
PASS ví dụ 3 -> -1
PASS một phần tử -> 0
PASS hai phần tử đã sắp xếp -> 0
PASS hai phần tử đảo -> 1
PASS nhỏ nhất ở cuối -> 1
PASS nhỏ nhất ở vị trí 1 -> 4
PASS đảo hoàn toàn -> -1
PASS hai chỗ giảm -> -1
PASS xoay giữa -> 3


Kiểm tay vài ca: `[2, 1]` dịch phải một lần thành `[1, 2]` ⇒ `1` ✓. `[5, 1, 2, 3, 4]`: phần tử nhỏ nhất `1` ở chỉ số 1, cần `5 − 1 = 4` lần dịch phải: lần 1 `[4,5,1,2,3]`, lần 2 `[3,4,5,1,2]`, lần 3 `[2,3,4,5,1]`, lần 4 `[1,2,3,4,5]` ✓. `[3, 1, 4, 2]`: giảm ở `3→1` và `4→2` ⇒ `-1` ✓.

### 12.2. Vét cạn mọi hoán vị nhỏ và ngẫu nhiên có seed

Input hợp lệ theo constraints: các số **khác nhau**, `1..100`, `n <= 100`. Với `n <= 7`, ta duyệt **mọi hoán vị** của `1..n`; với `n` lớn, ta lấy mẫu ngẫu nhiên (cả mảng xoay từ mảng tăng — luôn có đáp án — lẫn hoán vị bất kỳ).

In [23]:
from itertools import permutations
import random

dem = 0
for n in range(1, 8):
    for perm in permutations(range(1, n + 1)):
        nums = list(perm)
        exp = min_shifts_bruteforce(nums)
        for k, f in impls.items():
            assert f(nums[:]) == exp, (nums, k)
        dem += 1
print("vét cạn", dem, "hoán vị với n <= 7: mọi cách khớp brute force")

random.seed(2855)
for _ in range(1500):
    n = random.randint(1, 100)
    nums = random.sample(range(1, 101), n)
    if random.random() < 0.5:
        s = sorted(nums)
        k = random.randint(0, n - 1)
        nums = s[n - k:] + s[:n - k] if k else s      # đã dịch phải k lần từ mảng tăng: đáp án là n - k (hoặc 0)
    exp = min_shifts_bruteforce(nums)
    for k2, f in impls.items():
        assert f(nums[:]) == exp, (nums, k2)
print("1500 mảng ngẫu nhiên: mọi cách khớp brute force")

vét cạn 5913 hoán vị với n <= 7: mọi cách khớp brute force


1500 mảng ngẫu nhiên: mọi cách khớp brute force


### 12.3. Kiểm tính chất

- Nếu `nums` là mảng tăng dịch phải `k` lần (`0 < k < n`), thì cần dịch thêm `n − k` lần để đưa về mảng tăng: đáp án `n − k`.
- Đáp án luôn `-1` hoặc trong `[0, n − 1]`.
- Khi đáp án `k != -1`, dịch phải đúng `k` lần cho mảng tăng.
- Không phá mảng đầu vào.

In [24]:
random.seed(28550)
for _ in range(500):
    n = random.randint(2, 50)
    s = sorted(random.sample(range(1, 101), n))
    k = random.randint(1, n - 1)
    shifted = rotate_right(s, k)
    assert Solution().minimumRightShifts(shifted) == n - k
    copy = shifted[:]
    r = Solution().minimumRightShifts(shifted)
    assert shifted == copy
    assert r == -1 or 0 <= r < n
    if r != -1:
        assert rotate_right(shifted, r) == s
print("tính chất đúng")

tính chất đúng


## 13. Các lỗi người mới hay gặp

### Lỗi 1: Nhầm dịch phải với dịch trái

In [25]:
def sai_dich_trai(nums):
    n = len(nums)
    for k in range(n):
        r = nums[k:] + nums[:k]               # sai: đây là dịch TRÁI k lần
        if is_sorted(r):
            return k
    return -1

print("sai :", sai_dich_trai([3, 4, 5, 1, 2]))             # 3 (là số lần dịch TRÁI)
print("đúng:", Solution().minimumRightShifts([3, 4, 5, 1, 2]))   # 2

sai : 3
đúng: 2


**Vì sao sai:** dịch trái `k` lần là `nums[k:] + nums[:k]`; dịch phải `k` lần là `nums[n − k:] + nums[:n − k]`. Dịch trái cần `p` lần (chỉ số phần tử nhỏ nhất) còn dịch phải cần `n − p` lần.

### Lỗi 2: Quên so cặp vòng `(n − 1, 0)`

In [26]:
def sai_khong_vong(nums):
    n = len(nums)
    descents, pos = 0, -1
    for i in range(n - 1):                    # sai: bỏ cặp (n - 1, 0)
        if nums[i] > nums[i + 1]:
            descents += 1
            pos = i
    if descents == 0:
        return 0
    if descents > 1:
        return -1
    return n - (pos + 1)

print("sai :", sai_khong_vong([3, 4, 5, 1, 2]), "| đúng:", Solution().minimumRightShifts([3, 4, 5, 1, 2]))
print("sai :", sai_khong_vong([2, 3, 1, 5]), "| đúng:", Solution().minimumRightShifts([2, 3, 1, 5]))

sai : 2 | đúng: 2
sai : 2 | đúng: -1


**Vì sao sai:** bỏ cặp vòng, ta chấp nhận cả `[2, 3, 1, 5]` (một chỗ giảm tuyến tính `3 → 1`) dù `5 > 2` ở cặp vòng nghĩa là có **hai** chỗ giảm vòng. Mảng này **không** xoay thành mảng tăng được. Phải kiểm `nums[n − 1] > nums[0]` bằng `(i + 1) % n`.

### Lỗi 3: Trả về `n − pos` hoặc `pos` thay vì `n − (pos + 1)`

In [27]:
def sai_off_by_one(nums):
    n = len(nums)
    descents, pos = 0, -1
    for i in range(n):
        if nums[i] > nums[(i + 1) % n]:
            descents += 1
            pos = i
    if descents == 0:
        return 0
    if descents > 1:
        return -1
    return n - pos                            # sai: lệch 1

print("sai :", sai_off_by_one([3, 4, 5, 1, 2]), "| đúng:", Solution().minimumRightShifts([3, 4, 5, 1, 2]))
print("sai :", sai_off_by_one([1, 3, 5]), "| đúng:", Solution().minimumRightShifts([1, 3, 5]))

sai : 3 | đúng: 2
sai : 1 | đúng: 0


**Vì sao sai:** chỗ giảm ở cặp `(pos, pos + 1)` nên phần tử nhỏ nhất ở chỉ số `pos + 1`, cần `n − (pos + 1)` lần. Công thức lệch một đơn vị còn làm mảng đã sắp xếp (`pos = n − 1`) trả `1` thay vì `0`.

### Lỗi 4: Quên trường hợp mảng đã sắp xếp

In [28]:
def sai_khong_xu_ly_da_sap_xep(nums):
    n = len(nums)
    for i in range(n - 1):
        if nums[i] > nums[i + 1]:
            rotated = nums[i + 1:] + nums[:i + 1]
            return n - (i + 1) if is_sorted(rotated) else -1
    return n                                  # sai: mảng đã tăng cần 0 lần, không phải n lần

print("sai :", sai_khong_xu_ly_da_sap_xep([1, 3, 5]), "| đúng:", Solution().minimumRightShifts([1, 3, 5]))

sai : 3 | đúng: 0


**Vì sao sai:** mảng đã tăng cần `0` lần dịch (dịch `n` lần chỉ quay về như cũ, không phải số tối thiểu).

### Lỗi 5 (nhỏ): `nums.index(min(nums))` khi có số trùng

Đề đảm bảo các số **khác nhau**, nên `index` cho vị trí duy nhất. Nếu tự mở rộng đề cho số trùng thì phần tử nhỏ nhất có thể xuất hiện nhiều lần và `index` chỉ cho lần đầu; cần xử lý khác.

In [29]:
print([1, 1, 2].index(min([1, 1, 2])))    # 0 (chỉ vị trí đầu tiên)

0


## 14. Mẹo làm bài nhanh & nhận dạng pattern

- Đề "xoay / dịch vòng để sắp xếp" ⇒ nghĩ ngay tới **mảng đã xoay**: trên vòng tròn có **đúng một chỗ giảm**.
- Mẫu "đếm số chỗ giảm trên vòng tròn": dùng `(i + 1) % n` để gộp cặp cuối-đầu.
- **Dịch phải `k` lần** = `a[n − k:] + a[:n − k]`; **dịch trái `k` lần** = `a[k:] + a[:k]`. Đừng nhầm.
- Mảng có thể xoay thành mảng tăng ⇔ gồm hai đoạn tăng liên tiếp, và phần tử cuối nhỏ hơn phần tử đầu (khi cả hai đoạn đều không rỗng).
- Dùng brute force `O(n²)` (thử mọi `k`) làm oracle, rồi kiểm công thức bằng vét cạn mọi hoán vị nhỏ.

## 15. Biến thể của bài

1. **Có số trùng** (LeetCode 2855 bản II, hoặc 1752 Check if Array Is Sorted and Rotated): vẫn đếm chỗ giảm vòng tròn (`>`), nhưng đáp án và số chỗ giảm cần xử lý tinh hơn.
2. **Dịch trái** thay vì phải: đáp án là `(pos + 1) % n`.
3. **Chỉ hỏi có sắp xếp được hay không** (trả `True` / `False`): `descents <= 1` (cẩn thận `n = 1`).
4. **Sắp xếp giảm dần** bằng dịch vòng: đổi chiều so sánh.
5. **Tìm số lần dịch tối thiểu cả hai hướng:** `min(k, n − k)`.
6. **Số lần dịch với `n` lớn / nhiều truy vấn:** tính `pos` một lần rồi trả lời theo `mod n`.

## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Thử mọi `k` | xoay `k` lần, kiểm tra | `O(n²)` | `O(n)` | oracle, người mới |
| 2. Vị trí min | ứng viên duy nhất `k = (n − p) % n` | `O(n)` | `O(n)` | khi muốn dịch thật rồi kiểm |
| 3. Đếm chỗ giảm vòng | đúng 1 chỗ giảm ⇒ `n − (pos + 1)` | `O(n)` | `O(1)` | **code nộp** |
| 4. Điểm gãy đầu tiên | tìm chỗ giảm đầu, xoay, kiểm | `O(n)` | `O(n)` | dễ nghĩ |

## 17. 5 bài tập vận dụng (dễ → khó)

**Bài 1 (đoán output).** Không chạy code, đoán: `[1, 2, 3, 4][-1:] + [1, 2, 3, 4][:-1]`, `(4 + 1) % 5`, `[3, 4, 5, 1, 2].index(1)`, `rotate_right([1, 2, 3], 1)`.

**Bài 2 (tính tay).** Với `nums = [4, 5, 6, 1, 2, 3]`: liệt kê từng lần dịch phải, tìm số lần dịch để sắp xếp; sau đó kiểm bằng cách đếm chỗ giảm vòng tròn.

**Bài 3 (điền TODO).** Hoàn thành `min_shifts_todo(nums)` theo Cách 3.

In [30]:
def min_shifts_todo(nums):
    # TODO 1: n = len(nums), descents = 0, pos = -1
    # TODO 2: duyệt i từ 0 tới n - 1
    # TODO 3: nếu nums[i] > nums[(i + 1) % n] thì descents += 1 và pos = i
    # TODO 4: descents == 0 -> 0 ; descents > 1 -> -1
    # TODO 5: ngược lại trả về n - (pos + 1)
    pass

# KHÔNG gọi hàm TODO ở đây để khi Run All không bị lỗi.

**Bài 4 (tự viết lại).** Viết `min_shifts_min_index2(nums)` bằng vị trí của phần tử nhỏ nhất (Cách 2) **không nhìn** code mẫu.

**Bài 5 (biến thể / mini challenge).** (a) Viết `min_left_shifts(nums)`: số lần **dịch trái** tối thiểu để sắp xếp (hoặc `-1`). (b) Viết `can_sort_by_rotation(nums)` trả `True` / `False` cho bài **có thể có số trùng** (LeetCode 1752): trả `True` nếu mảng có thể xoay thành mảng không giảm.

## 18. Gợi ý 3 mức cho bài tập

- **Mức 1 (nhẹ):** Bài 1: `[1,2,3,4][-1:] = [4]`, `[:-1] = [1,2,3]`; `(4 + 1) % 5 = 0`; `[3,4,5,1,2].index(1) = 3`; `rotate_right([1,2,3], 1) = [3,1,2]`. Bài 2: phần tử nhỏ nhất `1` ở chỉ số 3, `n = 6`.
- **Mức 2 (vừa):** Bài 3: năm dòng đã ghi trong comment TODO. Bài 4: `p = nums.index(min(nums))`, `k = (n - p) % n`, xoay rồi kiểm `is_sorted`.
- **Mức 3 (sâu):** Bài 5(a): dịch trái `k` lần là `nums[k:] + nums[:k]`; với một chỗ giảm tại `pos`, `k = (pos + 1) % n`. Bài 5(b): với số trùng, chỗ giảm là `nums[i] > nums[(i + 1) % n]` (nghiêm ngặt) và điều kiện là `descents <= 1`.

## 19. Đáp án tham khảo cho bài tập

Chạy kiểm tra bằng `assert`.

In [31]:
# Bài 1
assert [1, 2, 3, 4][-1:] + [1, 2, 3, 4][:-1] == [4, 1, 2, 3]
assert (4 + 1) % 5 == 0
assert [3, 4, 5, 1, 2].index(1) == 3
assert rotate_right([1, 2, 3], 1) == [3, 1, 2]

# Bài 2: [4,5,6,1,2,3]
# 1 lần: [3,4,5,6,1,2]; 2 lần: [2,3,4,5,6,1]; 3 lần: [1,2,3,4,5,6]  -> 3
assert Solution().minimumRightShifts([4, 5, 6, 1, 2, 3]) == 3
assert rotate_right([4, 5, 6, 1, 2, 3], 3) == [1, 2, 3, 4, 5, 6]

# Bài 3
def min_shifts_todo_ans(nums):
    n = len(nums)
    descents, pos = 0, -1
    for i in range(n):
        if nums[i] > nums[(i + 1) % n]:
            descents += 1
            pos = i
    if descents == 0:
        return 0
    if descents > 1:
        return -1
    return n - (pos + 1)

assert min_shifts_todo_ans([3, 4, 5, 1, 2]) == 2
assert min_shifts_todo_ans([1, 3, 5]) == 0
assert min_shifts_todo_ans([2, 1, 4]) == -1

# Bài 4
def min_shifts_min_index2(nums):
    n = len(nums)
    p = nums.index(min(nums))
    k = (n - p) % n
    return k if is_sorted(rotate_right(nums, k)) else -1

assert min_shifts_min_index2([3, 4, 5, 1, 2]) == 2
assert min_shifts_min_index2([2, 1, 4]) == -1

# Bài 5(a): dịch trái
def min_left_shifts(nums):
    n = len(nums)
    for k in range(n):
        if is_sorted(nums[k:] + nums[:k]):
            return k
    return -1

assert min_left_shifts([3, 4, 5, 1, 2]) == 3
assert min_left_shifts([1, 3, 5]) == 0
assert min_left_shifts([2, 1, 4]) == -1

# Bài 5(b): có thể có số trùng
def can_sort_by_rotation(nums):
    n = len(nums)
    descents = sum(1 for i in range(n) if nums[i] > nums[(i + 1) % n])
    return descents <= 1

assert can_sort_by_rotation([3, 4, 5, 1, 2]) is True
assert can_sort_by_rotation([3, 4, 1, 2]) is True
assert can_sort_by_rotation([2, 1, 3, 4]) is False
assert can_sort_by_rotation([1, 1, 1]) is True
assert can_sort_by_rotation([1, 3, 2]) is False
assert can_sort_by_rotation([5]) is True

random.seed(19)
for _ in range(300):
    n = random.randint(1, 8)
    nums = random.sample(range(1, 30), n)
    assert min_shifts_todo_ans(nums) == min_shifts_min_index2(nums) == min_shifts_bruteforce(nums)
    assert (min_left_shifts(nums) != -1) == (min_shifts_bruteforce(nums) != -1)
    assert can_sort_by_rotation(nums) == (min_shifts_bruteforce(nums) != -1)
print("Tất cả đáp án đúng")

Tất cả đáp án đúng


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template** (xoay mảng để sắp xếp ⇒ đếm chỗ giảm trên vòng tròn)

```python
from typing import List

class Solution:
    def minimumRightShifts(self, nums: List[int]) -> int:
        n = len(nums)
        descents = 0
        pos = -1
        for i in range(n):
            if nums[i] > nums[(i + 1) % n]:       # cặp vòng tròn
                descents += 1
                pos = i
        if descents == 0:                          # n == 1
            return 0
        if descents > 1:                           # không xoay thành mảng tăng
            return -1
        return n - (pos + 1)                       # số lần dịch PHẢI
```

**Checklist**

- [ ] Phân biệt dịch **phải** (`a[n − k:] + a[:n − k]`) với dịch trái (`a[k:] + a[:k]`)?
- [ ] So cả cặp vòng `(n − 1, 0)` bằng `(i + 1) % n`?
- [ ] Số chỗ giảm vòng tròn phải **đúng 1** (hoặc `n = 1`), nếu không trả `-1`?
- [ ] Công thức `n − (pos + 1)` (không phải `n − pos`), mảng đã tăng cho `0`?
- [ ] Đã thử: một phần tử, đã sắp xếp, nhỏ nhất ở cuối / ở vị trí 1, hai chỗ giảm?
- [ ] Đã vét cạn mọi hoán vị nhỏ so với brute force?
- [ ] Signature đúng `minimumRightShifts(self, nums: List[int]) -> int`?